In [2]:
from pyspark.sql import functions as F

taxi = spark.table("silver_nyc_taxi_yellow")

taxi_clean = (
    taxi
    .withColumn("year", F.col("year").cast("int"))
    .withColumn("month", F.col("month").cast("int"))
    .withColumn("pickup_ts", F.col("pickup_ts").cast("timestamp"))
    .withColumn("dropoff_ts", F.col("dropoff_ts").cast("timestamp"))
    .withColumn(
        "trip_minutes",
        (F.col("dropoff_ts").cast("long") - F.col("pickup_ts").cast("long")) / 60.0
    )
    .filter(F.col("pickup_ts").isNotNull() & F.col("dropoff_ts").isNotNull())
    .filter(F.col("dropoff_ts") >= F.col("pickup_ts"))
    .filter(F.col("trip_distance").isNotNull() & (F.col("trip_distance") >= 0) & (F.col("trip_distance") <= 200))
    .filter(F.col("trip_minutes").isNotNull() & (F.col("trip_minutes") >= 0) & (F.col("trip_minutes") <= 600))
)

taxi_monthly = (
    taxi_clean
    .groupBy("year", "month")
    .agg(
        F.count("*").alias("trip_count"),
        F.sum("total_amount").alias("sum_total_amount"),
        F.sum("fare_amount").alias("sum_fare_amount"),
        F.sum("tip_amount").alias("sum_tip_amount"),
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.avg("trip_minutes").alias("avg_trip_minutes"),
        F.avg("passenger_count").alias("avg_passenger_count"),
    )
)


StatementMeta(, 719f90eb-a179-485f-8842-50d90fc739f0, 4, Finished, Available, Finished)

In [3]:
print(spark.table("silver_nyc_taxi_yellow").columns)

StatementMeta(, 719f90eb-a179-485f-8842-50d90fc739f0, 5, Finished, Available, Finished)

['vendor_id', 'pickup_ts', 'dropoff_ts', 'passenger_count', 'trip_distance', 'pu_location_id', 'do_location_id', 'payment_type', 'fare_amount', 'tip_amount', 'total_amount', 'year', 'month']


In [4]:
from pyspark.sql import functions as F

# 1) NYC Taxi monthly metrics (from SILVER)
taxi = spark.table("silver_nyc_taxi_yellow")

taxi_clean = (
    taxi
    .withColumn("year", F.col("year").cast("int"))
    .withColumn("month", F.col("month").cast("int"))
    .withColumn("pickup_ts", F.col("pickup_ts").cast("timestamp"))
    .withColumn("dropoff_ts", F.col("dropoff_ts").cast("timestamp"))
    .withColumn("trip_minutes", (F.col("dropoff_ts").cast("long") - F.col("pickup_ts").cast("long")) / 60.0)
    .filter(F.col("pickup_ts").isNotNull() & F.col("dropoff_ts").isNotNull())
    .filter(F.col("dropoff_ts") >= F.col("pickup_ts"))
    .filter(F.col("trip_distance").isNotNull() & (F.col("trip_distance") >= 0) & (F.col("trip_distance") <= 200))
    .filter(F.col("trip_minutes").isNotNull() & (F.col("trip_minutes") >= 0) & (F.col("trip_minutes") <= 600))
)

taxi_monthly = (
    taxi_clean
    .groupBy("year", "month")
    .agg(
        F.count("*").alias("trip_count"),
        F.sum("total_amount").alias("total_amount_usd"),
        F.sum("fare_amount").alias("fare_amount_usd"),
        F.sum("tip_amount").alias("tip_amount_usd"),
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.avg("trip_minutes").alias("avg_trip_minutes"),
        F.avg("passenger_count").alias("avg_passenger_count"),
    )
)

# 2) FX monthly averages (from BRONZE ECB)
fx = spark.table("bronze_ecb_fx")

fx_monthly = (
    fx
    .withColumn("year", F.year("date").cast("int"))
    .withColumn("month", F.month("date").cast("int"))
    .groupBy("year", "month")
    .agg(
        F.avg("usd_per_eur").alias("avg_usd_per_eur"),
        F.avg("eur_per_usd").alias("avg_eur_per_usd"),
    )
)

# 3) Join + (optional) convert USD totals to EUR using avg_eur_per_usd
gold_monthly = (
    taxi_monthly
    .join(fx_monthly, on=["year", "month"], how="left")
    .withColumn("total_amount_eur", F.col("total_amount_usd") * F.col("avg_eur_per_usd"))
    .withColumn("fare_amount_eur",  F.col("fare_amount_usd")  * F.col("avg_eur_per_usd"))
    .withColumn("tip_amount_eur",   F.col("tip_amount_usd")   * F.col("avg_eur_per_usd"))
)

display(gold_monthly.orderBy("year","month").limit(20))
print("GOLD monthly rows:", gold_monthly.count())


StatementMeta(, 719f90eb-a179-485f-8842-50d90fc739f0, 6, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, ca397801-ba68-41eb-8712-5e2573455cc2)

GOLD monthly rows: 12


In [5]:
spark.sql("DROP TABLE IF EXISTS gold_nyc_taxi_monthly")

gold_monthly.write.format("delta").mode("overwrite").saveAsTable("gold_nyc_taxi_monthly")

display(spark.table("gold_nyc_taxi_monthly").orderBy("year","month").limit(20))
print("GOLD table rows:", spark.table("gold_nyc_taxi_monthly").count())

StatementMeta(, 719f90eb-a179-485f-8842-50d90fc739f0, 7, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 88a451e6-3c7f-4edd-b8a1-61e20be76442)

GOLD table rows: 12


In [6]:
from pyspark.sql import functions as F

gold = spark.table("gold_nyc_taxi_monthly")

gdp_usa = (
    spark.table("silver_worldbank_gdp")
    .filter(F.col("country_iso3") == "USA")
    .select(F.col("year").cast("int").alias("year"), F.col("gdp_usd").cast("double").alias("usa_gdp_usd"))
)

gold_v2 = (
    gold.join(gdp_usa, on="year", how="left")
        .orderBy("year", "month")
)

display(gold_v2.limit(20))
print("Rows:", gold_v2.count())

StatementMeta(, 719f90eb-a179-485f-8842-50d90fc739f0, 8, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, ab546856-3e63-4943-bb69-098d60773626)

Rows: 12


In [7]:
spark.sql("DROP TABLE IF EXISTS gold_nyc_taxi_monthly_v2")

gold_v2.write.format("delta").mode("overwrite").saveAsTable("gold_nyc_taxi_monthly_v2")

display(spark.table("gold_nyc_taxi_monthly_v2").orderBy("year","month").limit(20))

StatementMeta(, 719f90eb-a179-485f-8842-50d90fc739f0, 9, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, a2b0502a-2f5f-44b0-8f87-ba469edbfae6)

In [8]:
from pyspark.sql import functions as F

gold2 = spark.table("gold_nyc_taxi_monthly_v2")

# check nulls
gold2.select(
    F.count("*").alias("rows"),
    F.sum(F.col("usa_gdp_usd").isNull().cast("int")).alias("gdp_nulls")
).show()

# show GDP by year (should make sense)
gold2.groupBy("year").agg(
    F.first("usa_gdp_usd").alias("usa_gdp_usd_example")
).orderBy("year").show(50)

# check how many months you have per year
gold2.groupBy("year").agg(
    F.countDistinct("month").alias("months_count")
).orderBy("year").show(50)

StatementMeta(, 719f90eb-a179-485f-8842-50d90fc739f0, 10, Finished, Available, Finished)

+----+---------+
|rows|gdp_nulls|
+----+---------+
|  12|        0|
+----+---------+

+----+-------------------+
|year|usa_gdp_usd_example|
+----+-------------------+
|2024|2.87509561307312E13|
+----+-------------------+

+----+------------+
|year|months_count|
+----+------------+
|2024|          12|
+----+------------+



In [9]:
spark.sql("DROP TABLE IF EXISTS gold_nyc_taxi_monthly_final")

spark.table("gold_nyc_taxi_monthly_v2") \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("gold_nyc_taxi_monthly_final")

StatementMeta(, 719f90eb-a179-485f-8842-50d90fc739f0, 11, Finished, Available, Finished)

In [10]:
spark.sql("DROP TABLE IF EXISTS gold_nyc_taxi_monthly_final")

spark.table("gold_nyc_taxi_monthly_v2") \
    .write.format("delta").mode("overwrite") \
    .saveAsTable("gold_nyc_taxi_monthly_final")

display(spark.table("gold_nyc_taxi_monthly_final"))

StatementMeta(, 719f90eb-a179-485f-8842-50d90fc739f0, 12, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 34191d10-0817-43ea-b0df-b7b92059dcfb)

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ---------- Sources ----------
taxi = spark.table("silver_nyc_taxi_yellow")          # has pickup_ts, dropoff_ts, pu_location_id, etc.
fx_raw = spark.table("bronze_ecb_fx")                # date, usd_per_eur, eur_per_usd
gdp_raw = spark.table("bronze_worldbank_gdp")         # country_name, countryiso3code, year, value

# ---------- Taxi -> DAILY aggregation (by pickup_date + pickup zone) ----------
taxi_clean = (
    taxi
    .withColumn("pickup_date", F.to_date("pickup_ts"))
    .withColumn("trip_minutes",
        (F.unix_timestamp("dropoff_ts") - F.unix_timestamp("pickup_ts")) / 60.0
    )
    .filter(F.col("dropoff_ts") >= F.col("pickup_ts"))
    .filter(F.col("trip_distance").isNotNull() & (F.col("trip_distance") >= 0) & (F.col("trip_distance") <= 200))
    .filter(F.col("trip_minutes").isNotNull() & (F.col("trip_minutes") >= 0) & (F.col("trip_minutes") <= 600))
)

taxi_daily = (
    taxi_clean
    .groupBy("pickup_date", F.col("pu_location_id").alias("zone_id"))
    .agg(
        F.count("*").alias("trip_count"),
        F.sum("total_amount").alias("total_amount_usd"),
        F.sum("fare_amount").alias("fare_amount_usd"),
        F.sum("tip_amount").alias("tip_amount_usd"),
        F.avg("trip_distance").alias("avg_trip_distance"),
        F.avg("trip_minutes").alias("avg_trip_minutes"),
        F.avg("passenger_count").alias("avg_passenger_count"),
    )
)

# ---------- Fill FX for ALL dates (weekends get previous available rate) ----------
min_max = taxi_daily.agg(F.min("pickup_date").alias("min_d"), F.max("pickup_date").alias("max_d")).collect()[0]
min_d, max_d = str(min_max["min_d"]), str(min_max["max_d"])

date_df = spark.sql(f"SELECT explode(sequence(to_date('{min_d}'), to_date('{max_d}'), interval 1 day)) as fx_date")

fx = fx_raw.select(F.col("date").alias("fx_date"), "usd_per_eur", "eur_per_usd")
w = Window.orderBy("fx_date").rowsBetween(Window.unboundedPreceding, 0)

fx_filled = (
    date_df.join(fx, "fx_date", "left")
    .withColumn("usd_per_eur", F.last("usd_per_eur", ignorenulls=True).over(w))
    .withColumn("eur_per_usd", F.last("eur_per_usd", ignorenulls=True).over(w))
)

# ---------- USA GDP (yearly) ----------
gdp_usa = (
    gdp_raw
    .select(
        F.col("countryiso3code").alias("country_iso3"),
        F.col("year").cast("int").alias("year"),
        F.col("value").cast("double").alias("usa_gdp_usd")
    )
    .filter(F.col("country_iso3") == "USA")
)

# ---------- Final GOLD FactTaxiDaily ----------
fact_taxi_daily = (
    taxi_daily
    .join(fx_filled, taxi_daily.pickup_date == fx_filled.fx_date, "left")
    .withColumn("year", F.year("pickup_date"))
    .join(gdp_usa, "year", "left")
    .withColumn("total_amount_eur", F.col("total_amount_usd") * F.col("eur_per_usd"))
    .withColumn("fare_amount_eur",  F.col("fare_amount_usd")  * F.col("eur_per_usd"))
    .withColumn("tip_amount_eur",   F.col("tip_amount_usd")   * F.col("eur_per_usd"))
    .drop("fx_date")
)

# ---------- Dimensions ----------
dim_date = (
    fact_taxi_daily.select(F.col("pickup_date").alias("date")).distinct()
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
    .select("date_key", "date", "year", "month", "day")
)

dim_zone = (
    fact_taxi_daily.select(F.col("zone_id").cast("int").alias("zone_key")).distinct()
)

dim_fx = (
    fx_filled
    .withColumn("date_key", F.date_format("fx_date", "yyyyMMdd").cast("int"))
    .select("date_key", F.col("fx_date").alias("date"), "usd_per_eur", "eur_per_usd")
)

dim_gdp = (
    gdp_usa.select(
        F.lit("USA").alias("country_iso3"),
        "year",
        "usa_gdp_usd"
    )
)

# ---------- Write GOLD tables (Lakehouse) ----------
spark.sql("DROP TABLE IF EXISTS gold_dim_date")
spark.sql("DROP TABLE IF EXISTS gold_dim_zone")
spark.sql("DROP TABLE IF EXISTS gold_dim_fx")
spark.sql("DROP TABLE IF EXISTS gold_dim_gdp")
spark.sql("DROP TABLE IF EXISTS gold_fact_taxi_daily")

dim_date.write.format("delta").mode("overwrite").saveAsTable("gold_dim_date")
dim_zone.write.format("delta").mode("overwrite").saveAsTable("gold_dim_zone")
dim_fx.write.format("delta").mode("overwrite").saveAsTable("gold_dim_fx")
dim_gdp.write.format("delta").mode("overwrite").saveAsTable("gold_dim_gdp")
fact_taxi_daily.write.format("delta").mode("overwrite").saveAsTable("gold_fact_taxi_daily")

display(spark.table("gold_fact_taxi_daily").limit(10))
print("gold_fact_taxi_daily rows:", spark.table("gold_fact_taxi_daily").count())

StatementMeta(, f96a2201-b12f-4a7f-b303-c8491cddb461, 3, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 9525fdc7-48ce-40fc-8b77-45078a873d96)

gold_fact_taxi_daily rows: 84947
